### Default hf dataloading

In [1]:
from transformers import default_data_collator
from transformers import (
    CONFIG_MAPPING,
    MODEL_FOR_CAUSAL_LM_MAPPING,
    AutoConfig,
    AutoModelForCausalLM,
    AutoTokenizer,
    HfArgumentParser,
    Trainer,
    TrainingArguments,
    default_data_collator,
    is_torch_xla_available,
    set_seed,
    get_scheduler,
)
import transformers
from itertools import chain

# from torchtune.models.llama3_2 import llama3_2_1b
from transformers.testing_utils import CaptureLogger
from torch.utils.data import DataLoader
from datasets import load_dataset

tok_logger = transformers.utils.logging.get_logger(
    "transformers.tokenization_utils_base"
)


raw_datasets = load_dataset(
    "HuggingFaceFW/fineweb-edu",
    name="sample-10BT",
    split="train",
    cache_dir="fineweb_edu_10b",
    num_proc=16,
)
raw_datasets = raw_datasets.select(list(range(len(raw_datasets) // 6)))

# column_names = list(raw_datasets["train"].features)
column_names = list(raw_datasets.features)
text_column_name = "text" if "text" in column_names else column_names[0]
block_size = 2048
model_name_or_path = "unsloth/Llama-3.2-1B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(
    model_name_or_path,
    use_fast=True,
)


def tokenize_function(examples):
    with CaptureLogger(tok_logger) as cl:
        output = tokenizer(examples[text_column_name])
    # clm input could be much much longer than block_size
    if "Token indices sequence length is longer than the" in cl.out:
        tok_logger.warning(
            "^^^^^^^^^^^^^^^^ Please ignore the warning above - this long input will be chunked into smaller bits"
            " before being passed to the model."
        )
    return output


tokenized_datasets = raw_datasets.map(
    tokenize_function,
    batched=True,
    num_proc=16,
    remove_columns=column_names,
    # load_from_cache_file=not data_args.overwrite_cache,
    desc="Running tokenizer on dataset",
)


def group_texts(examples):
    # Concatenate all texts.
    concatenated_examples = {k: list(chain(*examples[k])) for k in examples.keys()}
    total_length = len(concatenated_examples[list(examples.keys())[0]])
    # We drop the small remainder, and if the total_length < block_size  we exclude this batch and return an empty dict.
    # We could add padding if the model supported it instead of this drop, you can customize this part to your needs.
    total_length = (total_length // block_size) * block_size
    # Split by chunks of max_len.
    result = {
        k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated_examples.items()
    }
    result["labels"] = result["input_ids"].copy()
    return result


lm_datasets = tokenized_datasets.map(
    group_texts,
    batched=True,
    num_proc=16,
    desc=f"Grouping texts in chunks of {block_size}",
)
lm_datasets = lm_datasets.remove_columns(
    column_names=[item for item in lm_datasets.features.keys() if item != "input_ids"]
)

Resolving data files:   0%|          | 0/2410 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/98 [00:00<?, ?it/s]

In [3]:
lm_datasets

Dataset({
    features: ['input_ids'],
    num_rows: 790587
})

In [ ]:
# train_dataset = lm_datasets["train"]
train_dataset = lm_datasets
batch_size = 32 * 2
train_dataloader = DataLoader(
    train_dataset,
    shuffle=True,
    collate_fn=default_data_collator,
    batch_size=batch_size,
    drop_last=True,
    num_workers=16,
    # persistent_workers=True,
    pin_memory=True,
)

In [12]:
item

{'input_ids': tensor([[ 7893, 32424,   320,  ...,   374,    11,   814],
         [ 1450,    11,   568,  ..., 24623,  1392,    13],
         [96763,  1990,   279,  ...,   539,  3242,  5962],
         ...,
         [ 3810,   279, 56230,  ...,  1002,  5650,     8],
         [12751,    13,  7839,  ...,   315,  7447, 31696],
         [  927,    13,   358,  ...,  9256,    13, 78710]])}

In [ ]:
from tqdm import tqdm

for i in range(5):
    for item in tqdm(train_dataloader):
        item
# ~1250 it\s batch 4
# ~750 it\s batch 16
# ~620 it\s batch 32
# ~360 it\s batch 64

### Mosaic streaming dataloading

In [14]:
from streaming import MDSWriter, StreamingDataset
from torch.utils.data import DataLoader

local_dir = "fineweb_edu_10b_numpy_mds_chunked"
batch_size = 64
dataset = StreamingDataset(
    local=local_dir,
    remote=local_dir,
    batch_size=batch_size,
    # batch_size=64,
    split=None,
    shuffle=True,
)
dataloader = DataLoader(
    dataset,
    batch_size=batch_size,
    pin_memory=True,
    num_workers=16,
    # persistent_workers=True,
)

In [ ]:
from tqdm import tqdm

for i in range(5):
    for item in tqdm(dataloader):
        item
# ~2300 it\s batch 4
# ~1550 it\s batch 16
# ~1230 it\s batch 32
# ~1000 it\s batch 64